In [ ]:
from pathlib import Path

vocab_path = Path(r"e:\Project_NguyenMinhVu_2211110063\Source\datasets\hanviet_vocab\vocab.txt")
text = vocab_path.read_text(encoding="utf-8")
vocab_path.write_text(text.lower(), encoding="utf-8")

print("Done lowercasing vocab.txt")

UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 15-16: invalid continuation byte

In [1]:
# ===========================================================
# IMPORTS
# ===========================================================
import gc
import re
import unicodedata
import argparse
import sys
import time
from pathlib import Path
from typing import Optional, Literal

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import networkx as nx

try:
    from vncorenlp import VnCoreNLP
    VNCORENLP_AVAILABLE = True
except ImportError:
    VNCORENLP_AVAILABLE = False

# ===========================================================
# ▶▶ CẤU HÌNH — chỉnh tại đây nếu không dùng CLI ◀◀
# ===========================================================
INPUT_PATH    = r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\test_data.xlsx"
OUTPUT_PATH   = r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\test_data_output.xlsx"
MODEL_PATH    = r"E:\Project_NguyenMinhVu_2211110063\Source\ai\Model Train\Model_DG_ver2\vit5_grade_summary"
VNCORENLP_JAR = r"E:\Project_NguyenMinhVu_2211110063\Source\ai\VnCoreNLP-master\VnCoreNLP-1.1.1.jar"

COL_CONTENT   = "content"      # cột văn bản gốc
COL_GRADE     = "grade"        # cột lớp học (để "" nếu file không có)
COL_OUT       = "summary"  # cột kết quả — ghi vào cột có sẵn trong file

DEFAULT_GRADE    = 5        # lớp mặc định khi không có cột grade
DEFAULT_MODE     = "sample"   # "sample" | "beam"
DEFAULT_LENGTH   = "long"   # "short"  | "medium" | "long"
CHECKPOINT_EVERY = 50       # lưu checkpoint sau mỗi N dòng

# ===========================================================
# CONSTANTS
# ===========================================================
WORD_LIMIT               = 768
MAX_WORDS                = 768
SMALL_MODEL_CHAR_THRESH  = 100
SMALL_MODEL_GRADE        = 1

# ===========================================================
# ABSTRACTER AGENT
# ===========================================================

class AbstracterAgent:
    """Fine-tuned ViT5 grade-based abstractive summarizer."""

    def __init__(
        self,
        model_path: str = MODEL_PATH,
        small_model_path: Optional[str] = None,
        max_input_len: int = 768,
        max_target_len: int = 256,
        num_beams: int = 4,
        no_repeat_ngram_size: int = 3,
        annotator=None,
        vncorenlp=None,
        model_simcse: Optional[SentenceTransformer] = None,
    ):
        self.model_path           = model_path
        self.small_model_path     = small_model_path
        self._small_tokenizer     = None
        self._small_model         = None
        self.max_input_len        = max_input_len
        self.max_target_len       = max_target_len
        self.num_beams            = num_beams
        self.no_repeat_ngram_size = no_repeat_ngram_size
        self.annotator            = annotator
        self.vncorenlp            = vncorenlp
        self.device               = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_simcse         = model_simcse or SentenceTransformer(
            "VoVanPhuc/sup-SimCSE-VietNamese-phobert-base"
        )
        self._load_model()

    # ── model loading ────────────────────────────────────────

    def _load_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_path, use_fast=False)
        self.model     = AutoModelForSeq2SeqLM.from_pretrained(self.model_path)
        self.model.to(self.device)
        self.model.eval()

    def _load_small_model(self):
        if not self.small_model_path or self._small_model is not None:
            return
        self._small_tokenizer = AutoTokenizer.from_pretrained(self.small_model_path, use_fast=False)
        self._small_model     = AutoModelForSeq2SeqLM.from_pretrained(self.small_model_path)
        self._small_model.to(self.device)
        self._small_model.eval()

    def _unload_small_model(self):
        if self._small_model is None:
            return
        del self._small_model
        self._small_model, self._small_tokenizer = None, None
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def _seq2seq_for_request(self, content: str, grade: int):
        if (
            self.small_model_path
            and grade == SMALL_MODEL_GRADE
            and len(content.strip()) < SMALL_MODEL_CHAR_THRESH
        ):
            self._load_small_model()
            return self._small_tokenizer, self._small_model, True
        return self.tokenizer, self.model, False

    # ── text utilities ───────────────────────────────────────

    def normalize_text(self, text: str) -> str:
        text = text.replace("_", " ")
        text = re.sub(r'\s+([,\.!?;:\)\]])', r'\1', text)
        text = re.sub(r'([,\.!?;:])(?!\s)(?!\d)', r'\1 ', text)
        text = re.sub(r'([\(\[])\s+', r'\1', text)
        return re.sub(r' +', ' ', text).strip()

    def sentence_split(self, text: str) -> list:
        if self.vncorenlp is None:
            sents = re.split(r'(?<=[.!?])\s+', text.strip())
            return [self.normalize_text(s) for s in sents if s.strip()]
        try:
            sents = []
            for sent in self.vncorenlp.annotate(text)["sentences"]:
                raw = " ".join(w["form"] for w in sent)
                sents.append(self.normalize_text(raw))
            return sents
        except Exception:
            sents = re.split(r'(?<=[.!?])\s+', text.strip())
            return [self.normalize_text(s) for s in sents if s.strip()]

    def count_words(self, text: str) -> int:
        return len(text.split())

    # ── extractive pre-processing (TextRank + LexRank + MMR) ─

    def _tfidf_sim(self, sentences):
        tfidf = TfidfVectorizer().fit_transform(sentences)
        return cosine_similarity(tfidf)

    def textrank(self, sentences):
        sim   = self._tfidf_sim(sentences)
        graph = nx.from_numpy_array(sim)
        return nx.pagerank(graph)

    def lexrank(self, sentences):
        sim    = self._tfidf_sim(sentences)
        scores = sim.sum(axis=1)
        return dict(enumerate(scores))

    def filter_by_ratio(self, sentences, ratio=0.7):
        tr       = self.textrank(sentences)
        lr       = self.lexrank(sentences)
        combined = {i: 0.5 * tr[i] + 0.5 * lr[i] for i in range(len(sentences))}
        k        = max(1, int(len(sentences) * ratio))
        top_ids  = sorted(combined, key=combined.get, reverse=True)[:k]
        return [sentences[i] for i in sorted(top_ids)]

    def phobert_scoring(self, sentences, full_text):
        sent_emb = self.model_simcse.encode(sentences, normalize_embeddings=True)
        doc_emb  = self.model_simcse.encode([full_text], normalize_embeddings=True)[0]
        return sent_emb @ doc_emb

    def mmr(self, sentences, scores, lambda_=0.8, top_k=5):
        sent_emb   = self.model_simcse.encode(sentences, normalize_embeddings=True)
        selected, candidates = [], list(range(len(sentences)))
        for _ in range(min(top_k, len(candidates))):
            mmr_scores = [
                (i, lambda_ * scores[i] - (1 - lambda_) * max(
                    [sent_emb[i] @ sent_emb[j] for j in selected], default=0
                ))
                for i in candidates
            ]
            best = max(mmr_scores, key=lambda x: x[1])[0]
            selected.append(best)
            candidates.remove(best)
        return [sentences[i] for i in sorted(selected)]

    def extract_summary(self, text: str, max_words: int,
                        filter_ratio=0.7, mmr_ratio=0.5, lambda_=0.8) -> str:
        sentences = self.sentence_split(text)
        if len(sentences) < 3:
            result = self.normalize_text(text)
            if max_words and self.count_words(result) > max_words:
                result = " ".join(result.split()[:max_words])
            return result
        filtered = self.filter_by_ratio(sentences, ratio=filter_ratio)
        scores   = self.phobert_scoring(filtered, text)
        top_k    = max(1, int(len(filtered) * mmr_ratio))
        while top_k >= 1:
            summary = " ".join(self.mmr(filtered, scores, lambda_=lambda_, top_k=top_k))
            if not max_words or self.count_words(summary) <= max_words:
                break
            top_k -= 1
        else:
            summary = " ".join(summary.split()[:max_words])
        return summary

    def smart_summary(self, text: str) -> str:
        """Cắt bớt đầu vào nếu quá dài để vừa context window."""
        if self.count_words(text) > WORD_LIMIT:
            return self.extract_summary(text, max_words=MAX_WORDS)
        return text

    # ── length helpers ───────────────────────────────────────

    def _count_words(self, text: str) -> int:
        return len(text.split()) if isinstance(text, str) else 0

    def _word_to_token_estimate(self, n: int) -> int:
        return int(n * 1.3) if n > 0 else 0

    def _clean_summary(self, summary: str) -> str:
        if not isinstance(summary, str):
            return ""
        last = summary.rfind(".")
        return summary[:last + 1].strip() if last != -1 else summary.strip()

    # ── post-processing ──────────────────────────────────────

    def vietnamese_text_normalization(self, text: str) -> str:
        text = unicodedata.normalize("NFC", text)
        text = re.sub(r"\s+", " ", text)
        text = re.sub(r"\s+([,.!?;:])", r"\1", text)
        text = re.sub(r"([,.!?;:])([^\s])", r"\1 \2", text)
        text = re.sub(
            r"([a-zàáạảãăắằẳẵặâấầẩẫậđèéẹẻẽêếềểễệìíịỉĩòóọỏõôốồổỗộơớờởỡợùúụủũưứừửữựỳýỵỷỹ])"
            r"([A-ZÀÁẠẢÃĂẮẰẲẴẶÂẤẦẨẪẬĐ])", r"\1 \2", text
        )
        words = [w.capitalize() if w.isupper() and len(w) > 2 else w for w in text.split()]
        sents = re.split(r'(?<=[.!?]) +', " ".join(words))
        return " ".join(s[0].upper() + s[1:] if s else s for s in sents).strip()

    def extract_entities(self, text: str) -> list:
        if self.annotator is None or not isinstance(text, str) or not text.strip():
            return []
        entities = set()
        try:
            for sent in self.annotator.annotate(text).get("sentences", []):
                cur = []
                for token in sent:
                    label = token.get("nerLabel", "O")
                    if label.startswith("B-"):
                        if cur:
                            entities.add(" ".join(cur))
                        cur = [token["form"]]
                    elif label.startswith("I-") and cur:
                        cur.append(token["form"])
                    else:
                        if cur:
                            entities.add(" ".join(cur))
                        cur = []
                if cur:
                    entities.add(" ".join(cur))
        except Exception:
            pass
        return list(entities)

    def correct_entities(self, summary: str, entities: list, content: str) -> str:
        if not isinstance(summary, str) or not summary.strip():
            return summary
        for entity in sorted(entities, key=len, reverse=True):
            raw = entity.strip().replace("_", " ").strip()
            if not raw:
                continue
            m = re.search(re.escape(raw), content, flags=re.IGNORECASE) if content else None
            canonical   = m.group(0) if m else " ".join(w.capitalize() for w in raw.split())
            tokens      = re.split(r"\s+", raw)
            pattern_str = (r"[_ ]*".join(re.escape(t) for t in tokens)
                           if len(tokens) > 1 else re.escape(tokens[0]))
            summary = re.compile(pattern_str, re.IGNORECASE).sub(canonical, summary)
        return re.sub(r"\s+", " ", summary.replace("_", " ")).strip()

    # ── public API ───────────────────────────────────────────

    def summarize(
        self,
        content: str,
        grade: int,
        max_input_len: Optional[int] = None,
        max_target_len: Optional[int] = None,
        mode: str = "sample",
        length_option: Literal["short", "medium", "long"] = "medium",
    ) -> str:
        if not content or not content.strip():
            return "Nội dung trống."

        max_input_len  = max_input_len  or self.max_input_len
        max_target_len = max_target_len or self.max_target_len

        total_words    = self._count_words(content)
        ratio          = {"short": 0.30, "long": 0.60}.get(length_option, 0.40)
        summary_words  = int(total_words * ratio)

        target_max = min(self._word_to_token_estimate(summary_words), max_target_len)
        target_max = max(target_max, 1)
        target_min = max(1, min(int(target_max * 0.3), target_max))

        tokenizer, seq2seq_model, used_small = self._seq2seq_for_request(content, grade)
        try:
            prompt = f"Tóm tắt văn bản cho học sinh lớp {grade}: {content}"
            inputs = tokenizer(
                prompt, return_tensors="pt", truncation=True, max_length=max_input_len
            ).to(self.device)

            with torch.no_grad():
                if mode == "beam":
                    output_ids = seq2seq_model.generate(
                        **inputs,
                        min_new_tokens=target_min,
                        max_new_tokens=target_max,
                        num_beams=self.num_beams,
                        length_penalty=1.2,
                        no_repeat_ngram_size=self.no_repeat_ngram_size,
                        early_stopping=True,
                    )
                else:
                    output_ids = seq2seq_model.generate(
                        **inputs,
                        min_new_tokens=target_min,
                        max_new_tokens=target_max,
                        do_sample=True,
                        top_k=50,
                        top_p=0.9,
                        temperature=0.8,
                        no_repeat_ngram_size=self.no_repeat_ngram_size,
                    )

            summary = tokenizer.decode(output_ids[0], skip_special_tokens=True)
            del output_ids, inputs

            summary  = self._clean_summary(summary)
            summary  = self.vietnamese_text_normalization(summary)
            entities = self.extract_entities(content)
            if entities:
                summary = self.correct_entities(summary, entities, content)
            return summary.strip()
        finally:
            if used_small:
                self._unload_small_model()

    def run(
        self,
        content: str,
        grade: int = 5,
        mode: str = "sample",
        length_option: Literal["short", "medium", "long"] = "medium",
    ) -> str:
        try:
            return self.summarize(self.smart_summary(content), grade,
                                  mode=mode, length_option=length_option)
        except Exception as e:
            return f"[Error] {e}"


# ===========================================================
# BATCH RUNNER
# ===========================================================

def parse_args():
    p = argparse.ArgumentParser(description="Batch abstractive summarization từ Excel")
    p.add_argument("--input",          default=INPUT_PATH)
    p.add_argument("--output",         default=OUTPUT_PATH)
    p.add_argument("--model",          default=MODEL_PATH)
    p.add_argument("--vncorenlp",      default=VNCORENLP_JAR)
    p.add_argument("--content-col",    default=COL_CONTENT)
    p.add_argument("--grade-col",      default=COL_GRADE)
    p.add_argument("--out-col",        default=COL_OUT)
    p.add_argument("--default-grade",  type=int, default=DEFAULT_GRADE)
    p.add_argument("--mode",           default=DEFAULT_MODE, choices=["sample", "beam"])
    p.add_argument("--length",         default=DEFAULT_LENGTH, choices=["short", "medium", "long"])
    p.add_argument("--checkpoint",     type=int, default=CHECKPOINT_EVERY)
    p.add_argument("--resume",         action="store_true",
                   help="Tiếp tục: bỏ qua dòng đã có kết quả trong file output")
    # parse_known_args bỏ qua các tham số lạ do Jupyter tự thêm (vd: --f=kernel-xxx.json)
    args, _ = p.parse_known_args()
    return args


def load_vncorenlp(jar_path: str):
    if not VNCORENLP_AVAILABLE:
        print("[INFO] Thư viện vncorenlp chưa cài — dùng fallback regex.")
        return None, None
    if not jar_path or not Path(jar_path).exists():
        print("[INFO] Không tìm thấy VnCoreNLP jar — dùng fallback regex.")
        return None, None
    try:
        obj = VnCoreNLP(jar_path, annotators="wseg,pos,ner", max_heap_size="-Xmx2g")
        print("[INFO] VnCoreNLP đã tải.")
        return obj, obj
    except Exception as e:
        print(f"[WARN] Lỗi VnCoreNLP: {e} — dùng fallback regex.")
        return None, None


def load_dataframe(args) -> pd.DataFrame:
    out_path = Path(args.output)
    if args.resume and out_path.exists():
        df = pd.read_excel(args.output, dtype=str)
        print(f"[INFO] Resume từ output: {args.output} ({len(df)} dòng)")
    else:
        df = pd.read_excel(args.input, dtype=str)
        print(f"[INFO] Đọc input: {args.input} ({len(df)} dòng)")
    if args.out_col not in df.columns:
        df[args.out_col] = ""
    return df


def run_batch(args):
    bar = "=" * 62
    print(bar)
    print("  BATCH ABSTRACTIVE SUMMARIZATION")
    print(bar)
    print(f"  Input        : {args.input}")
    print(f"  Output       : {args.output}")
    print(f"  Mode/Length  : {args.mode} / {args.length}")
    print(f"  Content col  : {args.content_col}")
    print(f"  Grade col    : {args.grade_col or '(không có)'}")
    print(f"  Output col   : {args.out_col}")
    print(f"  Default grade: {args.default_grade}")
    print(f"  Resume       : {args.resume}")
    print(bar)

    annotator, vncorenlp = load_vncorenlp(args.vncorenlp)

    print("[INFO] Đang tải model ViT5 và SimCSE...")
    agent = AbstracterAgent(
        model_path  = args.model,
        annotator   = annotator,
        vncorenlp   = vncorenlp,
    )
    print(f"[INFO] Thiết bị: {agent.device}\n")

    df = load_dataframe(args)

    if args.content_col not in df.columns:
        sys.exit(
            f"[ERROR] Cột '{args.content_col}' không tồn tại.\n"
            f"  Các cột hiện có: {list(df.columns)}"
        )

    total     = len(df)
    processed = skipped = errors = 0
    t0        = time.time()

    for idx in range(total):
        row = df.iloc[idx]

        # Resume: bỏ qua dòng đã có kết quả
        if args.resume:
            val = str(row.get(args.out_col, "")).strip()
            if val and val.lower() not in ("", "nan"):
                skipped += 1
                continue

        content = str(row.get(args.content_col, "")).strip()
        if not content or content.lower() == "nan":
            df.at[idx, args.out_col] = ""
            continue

        # Lấy grade
        grade = args.default_grade
        if args.grade_col and args.grade_col in df.columns:
            try:
                grade = int(float(str(row[args.grade_col])))
            except (ValueError, TypeError):
                grade = args.default_grade

        # Tóm tắt
        try:
            result = agent.run(content, grade=grade,
                               mode=args.mode, length_option=args.length)
        except Exception as e:
            result = f"[ERROR] {e}"
            errors += 1

        df.at[idx, args.out_col] = result
        processed += 1

        # Progress log
        elapsed = time.time() - t0
        avg_sec = elapsed / processed
        eta_min = avg_sec * (total - skipped - processed) / 60
        print(
            f"  [{skipped + processed:>5}/{total}] row={idx} | lớp {grade} | "
            f"{len(content)}→{len(result)} ký tự | ETA {eta_min:.1f} phút",
            flush=True,
        )

        # Checkpoint
        if processed % args.checkpoint == 0:
            df.to_excel(args.output, index=False)
            print(f"  ✔ Checkpoint → {args.output}")

    # Lưu cuối
    df.to_excel(args.output, index=False)

    total_min = (time.time() - t0) / 60
    print(f"\n{bar}")
    print("  HOÀN TẤT")
    print(f"  Đã xử lý  : {processed} dòng")
    print(f"  Bỏ qua    : {skipped} dòng")
    print(f"  Lỗi       : {errors} dòng")
    print(f"  Thời gian : {total_min:.1f} phút")
    print(f"  Lưu tại   : {args.output}")
    print(bar)
    print("Van ban goc: ", content)
    print("Ban tom tat: ", result)

    if annotator:
        try:
            annotator.close()
        except Exception:
            pass


# ===========================================================
# ENTRY POINT 
# ===========================================================
if __name__ == "__main__":
    run_batch(parse_args())

c:\Users\minhv\anaconda3\envs\pytorch-env\lib\site-packages\networkx\utils\backends.py:135: RuntimeWarning: networkx backend defined more than once: nx-loopback
  backends.update(_get_backends("networkx.backends"))


  BATCH ABSTRACTIVE SUMMARIZATION
  Input        : E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\test_data.xlsx
  Output       : E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\test_data_output.xlsx
  Mode/Length  : sample / long
  Content col  : content
  Grade col    : grade
  Output col   : summary
  Default grade: 5
  Resume       : False
[INFO] VnCoreNLP đã tải.
[INFO] Đang tải model ViT5 và SimCSE...


No sentence-transformers model found with name VoVanPhuc/sup-SimCSE-VietNamese-phobert-base. Creating a new one with mean pooling.


[INFO] Thiết bị: cuda

[INFO] Đọc input: E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\test_data.xlsx (1 dòng)
  [    1/1] row=0 | lớp 5 | 849→527 ký tự | ETA 0.0 phút

  HOÀN TẤT
  Đã xử lý  : 1 dòng
  Bỏ qua    : 0 dòng
  Lỗi       : 0 dòng
  Thời gian : 0.1 phút
  Lưu tại   : E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\test_data_output.xlsx
Van ban goc:  Bảo vệ môi trường là trách nhiệm của mỗi người chúng ta, kể cả các bạn nhỏ. Môi trường xung quanh chúng ta đang ngày càng bị ô nhiễm do nhiều nguyên nhân khác nhau. Rác thải bị vứt bừa bãi ra đường phố, xuống sông hồ làm ô nhiễm nguồn nước và đất đai. Khói từ các nhà máy và phương tiện giao thông làm ô nhiễm không khí. Việc chặt phá rừng bừa bãi làm mất đi mái nhà của nhiều loài động vật và làm trái đất nóng lên. Mỗi chúng ta có thể làm nhiều việc nhỏ để bảo vệ môi trường như không vứt rác bừa bãi mà bỏ vào thùng rác, tiết kiệm điện và nước, trồng cây xanh quanh nhà và trường học, 

In [ ]:
from pathlib import Path
import re
import pandas as pd
from underthesea import word_tokenize
from bert_score import score as bertscore_fn
import torch 

HANVIET_VOCAB_PATH = Path(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\hanviet_vocab\vocab_mapped.xlsx"
)
GRADE_VOCAB_DIR = Path(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\grade_vocab"
)
_HANVIET_VOCAB_CACHE = None
_GRADE_VOCAB_CACHE = {}

def preprocess_vi_eval(text):
    text = text.strip()
    text = text.replace("\n", " ")
    tokens = word_tokenize(text, format="text")
    return tokens

def compute_bertscore_eval(candidate, reference):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    P, R, F1 = bertscore_fn(
        [candidate],
        [reference],
        lang="vi",
        model_type=None,
        device=device
    )

    return {
        "bertscore_precision": P.item(),
        "bertscore_recall": R.item(),
        "bertscore_f1": F1.item()
    }

def _normalize_token(token: str) -> str:
    token = token.strip().lower()
    token = re.sub(r"^[^\wÀ-ỹ]+|[^\wÀ-ỹ]+$", "", token, flags=re.UNICODE)
    return token


def load_hanviet_vocab(vocab_path: Path = HANVIET_VOCAB_PATH) -> set[str]:
    global _HANVIET_VOCAB_CACHE
    if _HANVIET_VOCAB_CACHE is not None:
        return _HANVIET_VOCAB_CACHE

    if not vocab_path.exists():
        _HANVIET_VOCAB_CACHE = set()
        return _HANVIET_VOCAB_CACHE

    df_vocab = pd.read_excel(vocab_path, dtype=str)
    if "ChuHanViet" not in df_vocab.columns:
        _HANVIET_VOCAB_CACHE = set()
        return _HANVIET_VOCAB_CACHE

    vocab = set()
    for word in df_vocab["ChuHanViet"].dropna().astype(str):
        w = _normalize_token(word)
        if w:
            vocab.add(w)

    _HANVIET_VOCAB_CACHE = vocab
    return _HANVIET_VOCAB_CACHE


def compute_hanviet_metrics(text: str, hanviet_vocab: set[str]) -> dict:
    if not isinstance(text, str) or not text.strip():
        return {
            "hanviet_word_count": 0,
            "hanviet_word_ratio": 0.0,
            "total_valid_tokens": 0,
        }

    tokens = preprocess_vi_eval(text).split()
    valid_tokens = []
    for tok in tokens:
        cleaned = _normalize_token(tok)
        if cleaned:
            valid_tokens.append(cleaned)

    total_valid_tokens = len(valid_tokens)
    if total_valid_tokens == 0:
        return {
            "hanviet_word_count": 0,
            "hanviet_word_ratio": 0.0,
            "total_valid_tokens": 0,
        }

    hanviet_word_count = sum(1 for tok in valid_tokens if tok in hanviet_vocab)
    hanviet_word_ratio = hanviet_word_count / total_valid_tokens

    return {
        "hanviet_word_count": hanviet_word_count,
        "hanviet_word_ratio": hanviet_word_ratio,
        "total_valid_tokens": total_valid_tokens,
    }


def load_grade_vocab(grade, vocab_dir: Path = GRADE_VOCAB_DIR) -> set[str]:
    if grade is None:
        return set()

    grade_str = str(grade).strip()
    match = re.search(r"\d+", grade_str)
    if not match:
        return set()

    grade_num = int(match.group())
    if grade_num in _GRADE_VOCAB_CACHE:
        return _GRADE_VOCAB_CACHE[grade_num]

    vocab_path = vocab_dir / f"grade_{grade_num}.txt"
    if not vocab_path.exists():
        _GRADE_VOCAB_CACHE[grade_num] = set()
        return _GRADE_VOCAB_CACHE[grade_num]

    vocab = set()
    for line in vocab_path.read_text(encoding="utf-8").splitlines():
        w = _normalize_token(line)
        if w:
            vocab.add(w)

    _GRADE_VOCAB_CACHE[grade_num] = vocab
    return _GRADE_VOCAB_CACHE[grade_num]


def compute_grade_vocab_metrics(text: str, grade) -> dict:
    if not isinstance(text, str) or not text.strip():
        return {
            "grade_vocab_word_count": 0,
            "grade_vocab_word_ratio": 0.0,
            "rare_word_count": 0,
            "summary_total_valid_tokens": 0,
        }

    tokens = preprocess_vi_eval(text).split()
    valid_tokens = []
    for tok in tokens:
        cleaned = _normalize_token(tok)
        if cleaned:
            valid_tokens.append(cleaned)

    total_valid_tokens = len(valid_tokens)
    if total_valid_tokens == 0:
        return {
            "grade_vocab_word_count": 0,
            "grade_vocab_word_ratio": 0.0,
            "rare_word_count": 0,
            "summary_total_valid_tokens": 0,
        }

    grade_vocab = load_grade_vocab(grade)
    grade_vocab_word_count = sum(1 for tok in valid_tokens if tok in grade_vocab)
    grade_vocab_word_ratio = grade_vocab_word_count / total_valid_tokens
    rare_word_count = total_valid_tokens - grade_vocab_word_count

    return {
        "grade_vocab_word_count": grade_vocab_word_count,
        "grade_vocab_word_ratio": grade_vocab_word_ratio,
        "rare_word_count": rare_word_count,
        "summary_total_valid_tokens": total_valid_tokens,
    }


def compute_difficulty_features_eval(text: str) -> dict:
    if not isinstance(text, str) or not text.strip():
        return {}

    raw = text.strip()
    sentence_splits = re.split(r"[.!?]+", raw)
    sentences = [s.strip() for s in sentence_splits if s.strip()]
    num_sentences = max(1, len(sentences))

    tokens = preprocess_vi_eval(raw).split()
    total_words = len(tokens)
    unique_words = len(set(tokens)) if total_words > 0 else 0
    type_token_ratio = (unique_words / total_words) if total_words > 0 else 0.0

    sentence_lengths = [len(preprocess_vi_eval(sent).split()) for sent in sentences]
    if sentence_lengths:
        avg_sentence_length = sum(sentence_lengths) / len(sentence_lengths)
        max_sentence_length = max(sentence_lengths)
        min_sentence_length = min(sentence_lengths)
        long_sentence_ratio = sum(1 for l in sentence_lengths if l >= 15) / len(sentence_lengths)
    else:
        avg_sentence_length = 0.0
        max_sentence_length = 0
        min_sentence_length = 0
        long_sentence_ratio = 0.0

    word_lengths = [len(w) for w in tokens] if tokens else []
    avg_word_length = (sum(word_lengths) / len(word_lengths)) if word_lengths else 0.0
    words_per_sentence = (total_words / num_sentences) if num_sentences > 0 else 0.0
    lexical_density = (unique_words / total_words) if total_words > 0 else 0.0

    if words_per_sentence <= 10 and avg_word_length <= 4:
        difficulty_level = "Grade 1"
        matched_rules = "grade1_simple_text"
    elif words_per_sentence <= 15 and avg_word_length <= 5:
        difficulty_level = "Grade 2"
        matched_rules = "grade2_basic_text"
    elif words_per_sentence <= 20:
        difficulty_level = "Grade 3"
        matched_rules = "grade3_intermediate_text"
    elif words_per_sentence <= 25:
        difficulty_level = "Grade 4"
        matched_rules = "grade4_advanced_text"
    else:
        difficulty_level = "Grade 5"
        matched_rules = "grade5_complex_text"

    hanviet_metrics = compute_hanviet_metrics(raw, load_hanviet_vocab())

    return {
        "difficulty_level": difficulty_level,
        "total_words": total_words,
        "unique_words": unique_words,
        "type_token_ratio": type_token_ratio,
        "num_sentences": num_sentences,
        "avg_sentence_length": avg_sentence_length,
        "max_sentence_length": max_sentence_length,
        "min_sentence_length": min_sentence_length,
        "long_sentence_ratio": long_sentence_ratio,
        "avg_word_length": avg_word_length,
        "words_per_sentence": words_per_sentence,
        "lexical_density": lexical_density,
        "matched_rules": matched_rules,
        **hanviet_metrics,
    }

def generate_comment_eval(results: dict) -> str:
    bert_f1 = results.get("bertscore_f1")
    comments = []
    if bert_f1 is not None:
        if bert_f1 > 0.8:
            comments.append("Mức độ tương đồng ngữ nghĩa với văn bản gốc rất cao.")
        elif bert_f1 > 0.6:
            comments.append("Mức độ tương đồng ngữ nghĩa với văn bản gốc ở mức trung bình.")
        else:
            comments.append("Mức độ tương đồng ngữ nghĩa với văn bản gốc còn thấp.")
    return " ".join(comments) if comments else ""

def evaluate_generated_summary(content: str, summary: str, grade=None):
    content_proc = preprocess_vi_eval(content)
    summary_proc = preprocess_vi_eval(summary)

    result = {}
    result2 = {}
    result.update(compute_bertscore_eval(summary_proc, content_proc))

    content_words = len(content_proc.split()) if content_proc else 0
    summary_words = len(summary_proc.split()) if summary_proc else 0
    result["content_words"] = content_words
    result["summary_words"] = summary_words
    result["compression_ratio"] = (summary_words / content_words) if content_words > 0 else 0.0

    result.update(compute_difficulty_features_eval(summary))
    result.update(compute_grade_vocab_metrics(summary, grade))
    result["comment"] = generate_comment_eval(result)

    result2.update(compute_difficulty_features_eval(content))
    return result, result2


df = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\test_data_output.xlsx")
df_grade = pd.read_excel(r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\dataset abstractive\test_data.xlsx")

content = df.iloc[0]["content"]
summary = df.iloc[0]["summary"]
grade = df.iloc[0]["grade"] if "grade" in df.columns else df_grade.iloc[0]["grade"]
metrics1, metrics2 = evaluate_generated_summary(content, summary, grade)
print(metrics1)
print(metrics2)

{'bertscore_precision': 0.9186726212501526, 'bertscore_recall': 0.8521157503128052, 'bertscore_f1': 0.8841433525085449, 'content_words': 154, 'summary_words': 97, 'compression_ratio': 0.6298701298701299, 'difficulty_level': 'Grade 3', 'total_words': 97, 'unique_words': 68, 'type_token_ratio': 0.7010309278350515, 'num_sentences': 6, 'avg_sentence_length': 15.166666666666666, 'max_sentence_length': 19, 'min_sentence_length': 12, 'long_sentence_ratio': 0.5, 'avg_word_length': 4.546391752577319, 'words_per_sentence': 16.166666666666668, 'lexical_density': 0.7010309278350515, 'matched_rules': 'grade3_intermediate_text', 'hanviet_word_count': 13, 'hanviet_word_ratio': 0.14942528735632185, 'total_valid_tokens': 87, 'grade_vocab_word_count': 86, 'grade_vocab_word_ratio': 0.9885057471264368, 'rare_word_count': 1, 'summary_total_valid_tokens': 87, 'comment': 'Mức độ tương đồng ngữ nghĩa với văn bản gốc rất cao.'}
{'difficulty_level': 'Grade 3', 'total_words': 154, 'unique_words': 103, 'type_toke

In [ ]:
import hashlib
import json
from langchain_community.chat_models import ChatOllama
import re
from pathlib import Path
import re
import pandas as pd
from underthesea import word_tokenize
from bert_score import score as bertscore_fn
import torch 

OLLAMA_MODEL = "qwen2.5:7b"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
_OLLAMA_CLIENTS = {}
_SYNONYM_CACHE = {}
_ADVERBIAL_CACHE = {}
HANVIET_VOCAB_PATH = Path(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\hanviet_vocab\vocab_mapped.xlsx"
)
GRADE_VOCAB_DIR = Path(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\grade_vocab"
)
_HANVIET_VOCAB_CACHE = None
_GRADE_VOCAB_CACHE = {}

def preprocess_vi_eval(text):
    text = text.strip()
    text = text.replace("\n", " ")
    tokens = word_tokenize(text, format="text")
    return tokens

def compute_bertscore_eval(candidate, reference):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    P, R, F1 = bertscore_fn(
        [candidate],
        [reference],
        lang="vi",
        model_type=None,
        device=device
    )

    return {
        "bertscore_precision": P.item(),
        "bertscore_recall": R.item(),
        "bertscore_f1": F1.item()
    }

def _normalize_token(token: str) -> str:
    token = token.strip().lower()
    token = re.sub(r"^[^\wÀ-ỹ]+|[^\wÀ-ỹ]+$", "", token, flags=re.UNICODE)
    return token


def load_hanviet_vocab(vocab_path: Path = HANVIET_VOCAB_PATH) -> set[str]:
    global _HANVIET_VOCAB_CACHE
    if _HANVIET_VOCAB_CACHE is not None:
        return _HANVIET_VOCAB_CACHE

    if not vocab_path.exists():
        _HANVIET_VOCAB_CACHE = set()
        return _HANVIET_VOCAB_CACHE

    df_vocab = pd.read_excel(vocab_path, dtype=str)
    if "ChuHanViet" not in df_vocab.columns:
        _HANVIET_VOCAB_CACHE = set()
        return _HANVIET_VOCAB_CACHE

    vocab = set()
    for word in df_vocab["ChuHanViet"].dropna().astype(str):
        w = _normalize_token(word)
        if w:
            vocab.add(w)

    _HANVIET_VOCAB_CACHE = vocab
    return _HANVIET_VOCAB_CACHE

def load_grade_vocab(grade, vocab_dir: Path = GRADE_VOCAB_DIR) -> set[str]:
    if grade is None:
        return set()

    grade_str = str(grade).strip()
    match = re.search(r"\d+", grade_str)
    if not match:
        return set()

    grade_num = int(match.group())
    if grade_num in _GRADE_VOCAB_CACHE:
        return _GRADE_VOCAB_CACHE[grade_num]

    vocab_path = vocab_dir / f"grade_{grade_num}.txt"
    if not vocab_path.exists():
        _GRADE_VOCAB_CACHE[grade_num] = set()
        return _GRADE_VOCAB_CACHE[grade_num]

    vocab = set()
    for line in vocab_path.read_text(encoding="utf-8").splitlines():
        w = _normalize_token(line)
        if w:
            vocab.add(w)

    _GRADE_VOCAB_CACHE[grade_num] = vocab
    return _GRADE_VOCAB_CACHE[grade_num]

def check_ollama(base_url: str = None, timeout: float = 3.0):
    import urllib.request
    u = (base_url or OLLAMA_BASE_URL).rstrip("/")
    try:
        urllib.request.urlopen(f"{u}/api/tags", timeout=timeout)
        return True, None
    except Exception as e:
        return False, repr(e)


def split_sentences_vi(paragraph: str):
    parts = re.split(r"([.!?]+)", paragraph)
    sentences = []
    for i in range(0, len(parts), 2):
        text = parts[i].strip() if i < len(parts) else ""
        punct = parts[i + 1] if i + 1 < len(parts) else ""
        if text:
            sentences.append((text + punct).strip())
    return sentences


def parse_list_response(raw: str):
    items = []
    for line in raw.splitlines():
        line = re.sub(r"^[-*\d\.)\s]+", "", line).strip().strip('"')
        if line:
            items.append(line)
    dedup = []
    seen = set()
    for x in items:
        k = _normalize_token(x.replace(" ", "_"))
        if k and k not in seen:
            seen.add(k)
            dedup.append(x)
    return dedup[:5]


def _get_chat_ollama(model: str = OLLAMA_MODEL, temperature: float = 0.0):
    key = (model, temperature, OLLAMA_BASE_URL)
    if key not in _OLLAMA_CLIENTS:
        _OLLAMA_CLIENTS[key] = ChatOllama(
            model=model,
            temperature=temperature,
            base_url=OLLAMA_BASE_URL,
        )
    return _OLLAMA_CLIENTS[key]


def _invoke_ollama(prompt: str, model: str = OLLAMA_MODEL, temperature: float = 0.0) -> str:
    llm = _get_chat_ollama(model=model, temperature=temperature)
    response = llm.invoke(prompt)
    if hasattr(response, "content"):
        return str(response.content).strip()
    return str(response).strip()


def _extract_json_obj(raw: str):
    raw = raw.strip()
    try:
        return json.loads(raw)
    except Exception:
        pass

    match = re.search(r"\{[\s\S]*\}", raw)
    if match:
        try:
            return json.loads(match.group(0))
        except Exception:
            return None
    return None


def _llm_synonym_vals_for_word(syn_map: dict, w: str):
    w = str(w).strip()
    if w in syn_map:
        v = syn_map[w]
    else:
        wn = _normalize_token(w.replace(" ", "_"))
        v = None
        for k, val in syn_map.items():
            if _normalize_token(str(k).replace(" ", "_")) == wn:
                v = val
                break
    if v is None:
        return []
    if isinstance(v, str):
        return [v]
    if isinstance(v, list):
        return v
    return []


def ask_ollama_lexical_synonyms_batch(sentence: str, words: list[str], model: str = OLLAMA_MODEL):
    unique_words = []
    seen = set()
    for w in words:
        w_clean = w.strip()
        if w_clean and w_clean not in seen:
            seen.add(w_clean)
            unique_words.append(w_clean)

    if not unique_words:
        return {"synonyms_map": {}}

    uncached_words = [w for w in unique_words if _normalize_token(w.replace(" ", "_")) not in _SYNONYM_CACHE]

    prompt = (
        f'Cho câu: "{sentence}"\n'
        "Các từ/cụm từ sau cần thay: hoặc là từ Hán Việt, hoặc là từ hiếm (khó, ít dùng so với mức lớp học) "
        "trong ngữ cảnh này. Hãy đề xuất cách nói thông tụ, đơn giản, dễ hiểu, đúng ngữ cảnh.\n"
        + "\n".join([f"- {w}" for w in unique_words])
        + "\nChỉ trả về JSON (không rút câu, không sửa phần khác):\n"
        + '{"synonyms":{"<từ gốc đúng như mục trên>":["gợi 1", ... tối đa 5]}}\n'
        + "Không giải thích."
    )

    try:
        raw = _invoke_ollama(prompt, model=model, temperature=0.2)
    except Exception as e:
        return {
            "synonyms_map": {},
            "error": f"ollama_invoke_failed: {e!r}",
            "raw_response": None,
        }

    data = _extract_json_obj(raw)
    if not isinstance(data, dict):
        data = {}

    data_synonyms = data.get("synonyms", {}) if isinstance(data, dict) else {}
    if not isinstance(data_synonyms, dict):
        data_synonyms = {}

    for w in uncached_words:
        key_norm = _normalize_token(w.replace(" ", "_"))
        vals = _llm_synonym_vals_for_word(data_synonyms, w)
        if isinstance(vals, str):
            vals = [vals]
        if not isinstance(vals, list):
            vals = []

        cleaned = []
        seen_c = set()
        for v in vals:
            v_str = str(v).strip()
            if not v_str:
                continue
            v_norm = _normalize_token(v_str.replace(" ", "_"))
            if v_norm and v_norm not in seen_c:
                seen_c.add(v_norm)
                cleaned.append(v_str)
        _SYNONYM_CACHE[key_norm] = cleaned[:5]

    synonyms_map = {
        w: _SYNONYM_CACHE.get(_normalize_token(w.replace(" ", "_")), [])
        for w in unique_words
    }
    out = {"synonyms_map": synonyms_map}
    if not any(synonyms_map.values()) and unique_words:
        out["warning"] = "llm_returned_empty_synonyms_check_json_keys"
        out["raw_response_preview"] = (raw[:800] + "…") if len(raw) > 800 else raw
    return out


def ask_ollama_adverbial_phrases(sentence: str, model: str = OLLAMA_MODEL):
    key = hashlib.md5(sentence.encode("utf-8")).hexdigest()
    if key in _ADVERBIAL_CACHE:
        return _ADVERBIAL_CACHE[key]

    prompt = (
        f'Cho câu: "{sentence}"\n'
        "Liệt kê các từ/cụm là TRẠNG NGỮ (thời gian, nơi chốn, mức độ, cách thức, điều kiện, "
        "mục đích phụ, nhận biết quan hệ 'trong/khi/tại/về phía...') có thể lược mà vẫn giữ ý chính.\n"
        "Bắt buộc xét cả cụm mở đầu câu kiểu: Trong quá trình..., Trong bối cảnh..., Với vai trò..., "
        "Ban đầu..., Hiện nay..., Nếu được..., (nếu chỉ là khung dẫn và có thể bỏ).\n"
        "Mỗi phần tử phải là đúng đoạn xuất hiện trong câu (kể cả dấu phẩy đi kèm nếu có).\n"
        "Chỉ trả về JSON: {\"adverbial_phrases\":[\"...\", ...]}. Không có thì []. Không giải thích."
    )
    raw = _invoke_ollama(prompt, model=model, temperature=0.0)
    data = _extract_json_obj(raw)
    if isinstance(data, dict) and isinstance(data.get("adverbial_phrases"), list):
        phrases = data["adverbial_phrases"]
    else:
        phrases = []
    if not isinstance(phrases, list):
        phrases = []
    out = [str(p).strip() for p in phrases if str(p).strip()]
    _ADVERBIAL_CACHE[key] = out
    return out


def _ws_norm(s: str) -> str:
    return re.sub(r"\s+", " ", (s or "").strip())


def _phrase_in_text(phrase: str, text: str) -> bool:
    ph = _ws_norm(phrase)
    if not ph:
        return False
    if ph in text:
        return True
    return ph in _ws_norm(text)


def remove_adverbial_phrases_from_text(text: str, phrases: list[str]):
    t = text
    ordered = []
    for p in phrases:
        s = str(p).strip()
        if s and s not in ordered:
            ordered.append(s)
    ordered.sort(key=len, reverse=True)
    removed = []
    for p in ordered:
        ws = _ws_norm(p)
        if not ws:
            continue
        if ws in t:
            t = t.replace(ws, " ", 1)
            removed.append(p)
            continue
        parts = ws.split()
        if not parts:
            continue
        pattern = r"\s+".join(re.escape(x) for x in parts)
        new_t, n = re.subn(pattern, " ", t, count=1, flags=re.UNICODE)
        if n > 0:
            t = new_t
            removed.append(p)
    t = re.sub(r"\s+", " ", t).strip()
    t = re.sub(r"([.!?])\s*,\s+", r"\1 ", t)
    t = re.sub(r"(?<=[.!?])\s+,", " ", t)
    return t, removed


def _cap_vi_first(s: str) -> str:
    s = s.strip()
    if not s:
        return s
    return s[0].upper() + s[1:]


def normalize_vi_orthography(text: str) -> str:
    t = (text or "").strip()
    if not t:
        return t
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"\s+([.!?;:,])", r"\1", t)
    t = re.sub(r"([.!?])\s*,\s+", r"\1 ", t)
    t = re.sub(r"([.!?;:])([A-Za-zÀ-ỹ])", r"\1 \2", t)
    t = re.sub(r",\s*", ", ", t)
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"\.{2,}", ".", t)
    parts = re.split(r"(?<=[.!?])\s+", t)
    out = []
    for p in parts:
        p = p.strip()
        if not p:
            continue
        p = re.sub(r"^,\s*", "", p)
        if not p:
            continue
        out.append(_cap_vi_first(p))
    t = " ".join(out) if out else t.strip()
    t = re.sub(r"\s+", " ", t).strip()
    return t


def ask_ollama_context_score(original_sentence: str, candidate_sentence: str, model: str = OLLAMA_MODEL):
    prompt = (
        "So sánh độ tự nhiên của 2 câu tiếng Việt trong cùng ngữ cảnh.\n"
        f"A: {original_sentence}\n"
        f"B: {candidate_sentence}\n"
        "Trả JSON duy nhất theo format: "
        '{"preferred":"A|B","confidence":0..1}. Không giải thích.'
    )
    raw = _invoke_ollama(prompt, model=model, temperature=0.0)

    try:
        data = json.loads(raw)
        preferred = data.get("preferred", "A")
        conf = float(data.get("confidence", 0.5))
        conf = min(max(conf, 0.0), 1.0)
    except Exception:
        preferred = "A"
        conf = 0.5

    if preferred == "B":
        return 0.5 + 0.5 * conf
    return 0.5 - 0.5 * conf


def infer_word_grade(word_norm: str, grade_vocab_map: dict):
    for g in range(1, 6):
        if word_norm in grade_vocab_map[g]:
            return g
    return 99


def identify_target_tokens(sentence: str, target_grade: int, grade_vocab_map: dict, hanviet_vocab: set):
    tokens = preprocess_vi_eval(sentence).split()
    targets = []
    target_grade_vocab = grade_vocab_map[target_grade]

    for tok in tokens:
        norm = _normalize_token(tok)
        if not norm:
            continue
        is_rare = norm not in target_grade_vocab
        is_hanviet = norm in hanviet_vocab
        if is_rare or is_hanviet:
            targets.append({
                "token": tok,
                "norm": norm,
                "is_rare": is_rare,
                "is_hanviet": is_hanviet,
            })
    return targets


def filter_candidates(candidates, target_grade: int, grade_vocab_map: dict):
    filtered = []
    for c in candidates:
        c_norm = _normalize_token(c.replace(" ", "_"))
        if not c_norm:
            continue
        g = infer_word_grade(c_norm, grade_vocab_map)
        if g != 99 and g <= target_grade:
            filtered.append(c.replace(" ", "_"))
    return filtered


def replace_token_in_sentence(sentence_tokenized: str, old_token: str, new_token: str):
    pattern = rf"(?<!\S){re.escape(old_token)}(?!\S)"
    return re.sub(pattern, new_token, sentence_tokenized, count=1)


def simplicity_score(old_norm: str, new_norm: str, target_grade: int, grade_vocab_map: dict, hanviet_vocab: set):
    old_grade = infer_word_grade(old_norm, grade_vocab_map)
    new_grade = infer_word_grade(new_norm, grade_vocab_map)
    score = 0.5
    if new_grade <= target_grade:
        score += 0.2
    if new_grade < old_grade:
        score += 0.2
    if old_norm in hanviet_vocab and new_norm not in hanviet_vocab:
        score += 0.1
    return min(score, 1.0)


def rank_candidate(original_sentence: str, candidate_sentence: str, old_norm: str, new_norm: str, target_grade: int, grade_vocab_map: dict, hanviet_vocab: set):
    try:
        sem = compute_bertscore_eval(
            preprocess_vi_eval(candidate_sentence),
            preprocess_vi_eval(original_sentence),
        )["bertscore_f1"]
    except Exception:
        sem = 0.0
    try:
        ctx = ask_ollama_context_score(original_sentence, candidate_sentence)
    except Exception:
        ctx = 0.5
    try:
        simp = simplicity_score(old_norm, new_norm, target_grade, grade_vocab_map, hanviet_vocab)
    except Exception:
        simp = 0.5
    total = 0.5 * sem + 0.3 * ctx + 0.2 * simp
    return {
        "semantic_score": sem,
        "context_score": ctx,
        "simplicity_score": simp,
        "total_score": total,
    }


def simplify_sentence(
    sentence: str,
    target_grade: int,
    max_replacements_per_sentence: int = 1,
    remove_adverbials: bool = True,
    min_total_score: float = 0.85,
    min_semantic_score: float = 0.9,
    normalize_orthography: bool = True,
):
    _ok, _err = check_ollama()
    debug_info = {
        "ollama_reachable": _ok,
        "ollama_check": _err,
        "lexical_batch_error": None,
        "lexical_batch_warning": None,
        "per_token": [],
    }

    grade_vocab_map = {g: load_grade_vocab(g) for g in range(1, 6)}
    hanviet_vocab = load_hanviet_vocab()

    temp = sentence
    all_marked = identify_target_tokens(temp, target_grade, grade_vocab_map, hanviet_vocab)
    replace_targets = [t for t in all_marked if t["is_hanviet"] or t["is_rare"]]
    debug_info["n_replace_targets"] = len(replace_targets)

    words_for_batch = [t["token"].replace("_", " ") for t in replace_targets]
    sentence_candidates_map = {}
    batch_result = {}
    if words_for_batch:
        try:
            batch_result = ask_ollama_lexical_synonyms_batch(temp, words_for_batch)
            sentence_candidates_map = batch_result.get("synonyms_map", {})
            debug_info["lexical_batch_error"] = batch_result.get("error")
            debug_info["lexical_batch_warning"] = batch_result.get("warning")
            debug_info["raw_preview"] = batch_result.get("raw_response_preview")
        except Exception as e:
            debug_info["lexical_batch_error"] = repr(e)
            sentence_candidates_map = {}

    normalized_candidates_map = {}
    for src_word, cands in sentence_candidates_map.items():
        src_norm = _normalize_token(str(src_word).replace(" ", "_"))
        if src_norm:
            normalized_candidates_map[src_norm] = cands

    current_tokenized = preprocess_vi_eval(temp)
    lexical_replacements = []

    for t in replace_targets:
        if len(lexical_replacements) >= max_replacements_per_sentence:
            break

        src_token = t["token"]
        src_norm = t["norm"]

        candidates = normalized_candidates_map.get(src_norm, [])
        if not candidates and src_token in sentence_candidates_map:
            candidates = sentence_candidates_map.get(src_token, [])

        filtered = filter_candidates(candidates, target_grade, grade_vocab_map)
        tok_dbg = {
            "token": src_token,
            "norm": src_norm,
            "n_raw": len(candidates),
            "n_filtered": len(filtered),
        }
        if not candidates:
            tok_dbg["reason"] = "no_synonyms_from_llm_or_cache"
        elif not filtered:
            tok_dbg["reason"] = "no_candidate_in_grade_vocab_1_to_target"
        if not filtered:
            debug_info["per_token"].append(tok_dbg)
            continue

        scored = []
        for c in filtered:
            cand_tokenized = replace_token_in_sentence(current_tokenized, src_token, c)
            cand_sentence = cand_tokenized.replace("_", " ")
            score_info = rank_candidate(
                current_tokenized.replace("_", " "),
                cand_sentence,
                src_norm,
                _normalize_token(c),
                target_grade,
                grade_vocab_map,
                hanviet_vocab,
            )
            scored.append({
                "old_word": src_token,
                "new_word": c,
                "candidate_sentence": cand_sentence,
                **score_info,
            })

        if not scored:
            tok_dbg["reason"] = "rank_failed"
            debug_info["per_token"].append(tok_dbg)
            continue

        best = max(scored, key=lambda x: x["total_score"])
        tok_dbg["best_total"] = round(best["total_score"], 4)
        tok_dbg["best_semantic"] = round(best["semantic_score"], 4)
        if best["total_score"] >= min_total_score and best["semantic_score"] >= min_semantic_score:
            current_tokenized = replace_token_in_sentence(current_tokenized, src_token, best["new_word"])
            lexical_replacements.append(best)
            tok_dbg["accepted"] = True
        else:
            tok_dbg["reason"] = "below_threshold"
        debug_info["per_token"].append(tok_dbg)

    temp_after_lexical = current_tokenized.replace("_", " ")

    adverbial_phrases_found = []
    adverbial_phrases_verified = []
    adverbial_phrases_removed = []
    final_sentence = temp_after_lexical

    if remove_adverbials:
        try:
            adverbial_phrases_found = ask_ollama_adverbial_phrases(temp_after_lexical)
        except Exception:
            adverbial_phrases_found = []
        adverbial_phrases_verified = [
            p for p in adverbial_phrases_found if p and _phrase_in_text(str(p).strip(), temp_after_lexical)
        ]
        final_sentence, adverbial_phrases_removed = remove_adverbial_phrases_from_text(
            temp_after_lexical, adverbial_phrases_verified
        )

    if normalize_orthography:
        final_sentence = normalize_vi_orthography(final_sentence)

    return {
        "original_sentence": sentence,
        "lexical_phrases_targeted": [
            {
                "token": t["token"],
                "norm": t["norm"],
                "is_hanviet": t["is_hanviet"],
                "is_rare": t["is_rare"],
            }
            for t in replace_targets
        ],
        "lexical_replacements": lexical_replacements,
        "sentence_after_lexical": temp_after_lexical,
        "adverbial_phrases_found": adverbial_phrases_found,
        "adverbial_phrases_verified": adverbial_phrases_verified,
        "adverbial_phrases_removed": adverbial_phrases_removed,
        "simplified_sentence": final_sentence,
        "diagnostics": debug_info,
    }


def simplify_paragraph(
    paragraph: str,
    target_grade: int,
    max_replacements_per_sentence: int = 1,
    remove_adverbials: bool = True,
    min_total_score: float = 0.9,
    min_semantic_score: float = 0.95,
    normalize_orthography: bool = True,
):
    sentences = split_sentences_vi(paragraph)
    outputs = []
    new_sentences = []

    for s in sentences:
        out = simplify_sentence(
            s,
            target_grade,
            max_replacements_per_sentence=max_replacements_per_sentence,
            remove_adverbials=remove_adverbials,
            min_total_score=min_total_score,
            min_semantic_score=min_semantic_score,
            normalize_orthography=normalize_orthography,
        )
        outputs.append(out)
        new_sentences.append(out["simplified_sentence"])

    return {
        "original_paragraph": paragraph,
        "simplified_paragraph": " ".join(new_sentences),
        "sentence_results": outputs,
    }

example_paragraph = """Trong quá trình học tập, mỗi học sinh cần có ý thức và trách nhiệm đối với việc phát triển bản thân. Việc tiếp thu kiến thức không chỉ giúp nâng cao năng lực mà còn góp phần hình thành tư duy logic và khả năng phân tích vấn đề. Bên cạnh đó, môi trường giáo dục đóng vai trò quan trọng trong việc định hướng và hỗ trợ học sinh đạt được mục tiêu của mình. Nếu biết tận dụng cơ hội và không ngừng nỗ lực, các em sẽ từng bước hoàn thiện kỹ năng và xây dựng nền tảng vững chắc cho tương lai."""
print("Ollama:", check_ollama())
result_simplify = simplify_paragraph(example_paragraph, target_grade=5, max_replacements_per_sentence=1, remove_adverbials=True, min_total_score=0.9, min_semantic_score=0.95, normalize_orthography=True)
print(result_simplify["simplified_paragraph"])
print(result_simplify["sentence_results"][0]["lexical_replacements"][:2])
print("diagnostics[0]:")
print(json.dumps(result_simplify["sentence_results"][0].get("diagnostics", {}), ensure_ascii=False, indent=2)[:4000])

Ollama: (True, None)
Mỗi học sinh cần có ý thức và trách nhiệm đối với việc phát triển bản thân. Việc tiếp thu kiến thức không chỉ giúp nâng cao năng lực mà còn góp phần hình thành tư duy logic và khả năng phân tích vấn đề. Môi trường giáo dục đóng vai trò quan trọng. Các em sẽ từng bước hoàn thiện kỹ năng và xây dựng bản lĩnh vững chắc cho tương lai.
[]
diagnostics[0]:
{
  "ollama_reachable": true,
  "ollama_check": null,
  "lexical_batch_error": null,
  "lexical_batch_warning": "llm_returned_empty_synonyms_check_json_keys",
  "per_token": [
    {
      "token": "cần",
      "norm": "cần",
      "n_raw": 0,
      "n_filtered": 0,
      "reason": "no_synonyms_from_llm_or_cache"
    }
  ],
  "n_replace_targets": 1,
  "raw_preview": "{\"synonyms\":{\"need\":[\"phải\", \"cần thiết\", \"phụ thuộc\", \"yêu cầu\", \"bắt buộc\"]}}"
}


Mỗi học sinh cần có ý thức và trách nhiệm đối với việc phát triển bản thân. Việc tiếp thu kiến thức không chỉ giúp nâng cao năng lực mà còn góp phần hình thành tư duy logic và khả năng phân tích vấn đề. Môi trường giáo dục đóng vai trò quan trọng. Các em sẽ từng bước hoàn thiện kỹ năng và xây dựng bản lĩnh vững chắc cho tương lai.

Trong quá trình học tập, mỗi học sinh cần có ý thức và trách nhiệm đối với việc phát triển bản thân. Việc tiếp thu kiến thức không chỉ giúp nâng cao năng lực mà còn góp phần hình thành tư duy logic và khả năng phân tích vấn đề. Bên cạnh đó, môi trường giáo dục đóng vai trò quan trọng trong việc định hướng và hỗ trợ học sinh đạt được mục tiêu của mình. Nếu biết tận dụng cơ hội và không ngừng nỗ lực, các em sẽ từng bước hoàn thiện kỹ năng và xây dựng nền tảng vững chắc cho tương lai.

In [ ]:
import json
import re
from langchain_community.chat_models import ChatOllama
from pathlib import Path
import re
import pandas as pd
from underthesea import word_tokenize
from bert_score import score as bertscore_fn
import torch 

DIFF_UPGRADE_MAX_RATIO = 0.3
DIFF_MIN_BERT_F1 = 0.9
DIFF_MAX_LENGTH_RATIO = 1.2
_UPGRADE_LLM = {}
UPGRADE_OLLAMA_BASE_URL = "http://127.0.0.1:11434"
OLLAMA_MODEL = "qwen2.5:7b"
OLLAMA_BASE_URL = "http://127.0.0.1:11434"
_OLLAMA_CLIENTS = {}
_SYNONYM_CACHE = {}
_ADVERBIAL_CACHE = {}
HANVIET_VOCAB_PATH = Path(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\hanviet_vocab\vocab_mapped.xlsx"
)
GRADE_VOCAB_DIR = Path(
    r"E:\Project_NguyenMinhVu_2211110063\Source\datasets\grade_vocab"
)
_HANVIET_VOCAB_CACHE = None
_GRADE_VOCAB_CACHE = {}

def preprocess_vi_eval(text):
    text = text.strip()
    text = text.replace("\n", " ")
    tokens = word_tokenize(text, format="text")
    return tokens

def compute_bertscore_eval(candidate, reference):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    P, R, F1 = bertscore_fn(
        [candidate],
        [reference],
        lang="vi",
        model_type=None,
        device=device
    )

    return {
        "bertscore_precision": P.item(),
        "bertscore_recall": R.item(),
        "bertscore_f1": F1.item()
    }

def _normalize_token(token: str) -> str:
    token = token.strip().lower()
    token = re.sub(r"^[^\wÀ-ỹ]+|[^\wÀ-ỹ]+$", "", token, flags=re.UNICODE)
    return token


def load_hanviet_vocab(vocab_path: Path = HANVIET_VOCAB_PATH) -> set[str]:
    global _HANVIET_VOCAB_CACHE
    if _HANVIET_VOCAB_CACHE is not None:
        return _HANVIET_VOCAB_CACHE

    if not vocab_path.exists():
        _HANVIET_VOCAB_CACHE = set()
        return _HANVIET_VOCAB_CACHE

    df_vocab = pd.read_excel(vocab_path, dtype=str)
    if "ChuHanViet" not in df_vocab.columns:
        _HANVIET_VOCAB_CACHE = set()
        return _HANVIET_VOCAB_CACHE

    vocab = set()
    for word in df_vocab["ChuHanViet"].dropna().astype(str):
        w = _normalize_token(word)
        if w:
            vocab.add(w)

    _HANVIET_VOCAB_CACHE = vocab
    return _HANVIET_VOCAB_CACHE

def load_grade_vocab(grade, vocab_dir: Path = GRADE_VOCAB_DIR) -> set[str]:
    if grade is None:
        return set()

    grade_str = str(grade).strip()
    match = re.search(r"\d+", grade_str)
    if not match:
        return set()

    grade_num = int(match.group())
    if grade_num in _GRADE_VOCAB_CACHE:
        return _GRADE_VOCAB_CACHE[grade_num]

    vocab_path = vocab_dir / f"grade_{grade_num}.txt"
    if not vocab_path.exists():
        _GRADE_VOCAB_CACHE[grade_num] = set()
        return _GRADE_VOCAB_CACHE[grade_num]

    vocab = set()
    for line in vocab_path.read_text(encoding="utf-8").splitlines():
        w = _normalize_token(line)
        if w:
            vocab.add(w)

    _GRADE_VOCAB_CACHE[grade_num] = vocab
    return _GRADE_VOCAB_CACHE[grade_num]


def _get_upgrade_llm(model: str = "qwen2.5:7b", temperature: float = 0.25):
    key = (model, temperature)
    if key not in _UPGRADE_LLM:
        _UPGRADE_LLM[key] = ChatOllama(
            model=model,
            temperature=temperature,
            base_url=UPGRADE_OLLAMA_BASE_URL,
        )
    return _UPGRADE_LLM[key]


def _invoke_upgrade_llm(prompt: str, model: str = "qwen2.5:7b", temperature: float = 0.25) -> str:
    llm = _get_upgrade_llm(model=model, temperature=temperature)
    r = llm.invoke(prompt)
    return str(r.content).strip() if hasattr(r, "content") else str(r).strip()


def _extract_json_upgrade(raw: str):
    raw = raw.strip()
    try:
        return json.loads(raw)
    except Exception:
        pass
    m = re.search(r"\{[\s\S]*\}", raw)
    if m:
        try:
            return json.loads(m.group(0))
        except Exception:
            return None
    return None


def _json_upgrade_as_dict(data):
    if isinstance(data, dict):
        return data
    if isinstance(data, list) and len(data) == 1 and isinstance(data[0], dict):
        return data[0]
    return {}


def _replacement_allowed_for_next_grade(replacement: str, vocab_next_grade: set[str]) -> bool:
    if not replacement or not vocab_next_grade:
        return False
    rep = replacement.strip()
    key_full = _normalize_token(rep.replace(" ", "_"))
    if key_full and key_full in vocab_next_grade:
        return True
    parts = [_normalize_token(x) for x in rep.split() if _normalize_token(x)]
    if not parts:
        return False
    return all(p in vocab_next_grade for p in parts)


def split_sentences_for_upgrade(paragraph: str):
    parts = re.split(r"([.!?]+)", paragraph)
    sentences = []
    for i in range(0, len(parts), 2):
        text = parts[i].strip() if i < len(parts) else ""
        punct = parts[i + 1] if i + 1 < len(parts) else ""
        if text:
            sentences.append((text + punct).strip())
    return sentences


def replace_span_once(text: str, old: str, new: str) -> str:
    pattern = rf"(?<!\S){re.escape(old)}(?!\S)"
    return re.sub(pattern, new, text, count=1)


def similarity_candidate_vs_reference(candidate: str, reference: str):
    ref = preprocess_vi_eval(reference)
    cand = preprocess_vi_eval(candidate)
    f1 = compute_bertscore_eval(cand, ref)["bertscore_f1"]
    ra = len(ref.split()) if ref else 1
    rb = len(cand.split()) if cand else 1
    lr = rb / ra if ra else 1.0
    return f1, lr


def ask_llm_nv_targets(sentence: str, ollama_model: str, max_targets: int = 12):
    prompt = (
        f'Cho câu sau (giữ nguyên ý, không sửa câu):\n"{sentence}"\n\n'
        "Chỉ liệt kê các **danh từ** và **động từ** (hoặc cụm danh từ/động từ ngắn) có thể đổi sang cách nói trang trọng hơn trong giáo dục. "
        "Mỗi mục \"span\" phải là đoạn xuất hiện **nguyên văn** trong câu (đúng khoảng trắng). Không liệt kê tên riêng nếu không chắc.\n"
        f"Tối đa {max_targets} mục.\n"
        "Trả JSON duy nhất:\n"
        '{"targets":[{"span":"...","pos":"noun|verb"}]}\n'
        "Không giải thích."
    )
    raw = _invoke_upgrade_llm(prompt, model=ollama_model, temperature=0.1)
    data = _json_upgrade_as_dict(_extract_json_upgrade(raw))
    targets = data.get("targets", [])
    if not isinstance(targets, list):
        return []
    out = []
    for t in targets:
        if not isinstance(t, dict):
            continue
        sp = str(t.get("span", "")).strip()
        pos = str(t.get("pos", "")).strip().lower()
        if not sp or sp not in sentence:
            continue
        if pos not in ("noun", "verb", "danh từ", "động từ"):
            pos = "word"
        out.append({"span": sp, "pos": pos})
    return out


def ask_llm_nv_targets_pair(sentence_a: str, sentence_b: str, ollama_model: str, max_targets: int = 14):
    prompt = (
        "Cho hai câu liên tiếp (giữ nguyên ý, không sửa nội dung trong khối sau):\n"
        f'Câu 1: "{sentence_a}"\n'
        f'Câu 2: "{sentence_b}"\n\n'
        "Liệt kê **danh từ** và **động từ** (hoặc cụm ngắn) có thể đổi sang cách nói trang trọng hơn trong giáo dục; "
        "mỗi \"span\" phải xuất hiện **nguyên văn** trong Câu 1 hoặc Câu 2.\n"
        "Đồng thời có thể đề xuất **tối đa một** cụm **trạng ngữ liên kết** đặt ngay đầu Câu 2 (sau dấu đầu câu nếu có) để hai câu mạch lạc hơn; "
        "nếu không cần thì để `link_prefix_sentence2` là chuỗi rỗng.\n"
        f"Tối đa {max_targets} mục trong targets.\n"
        "Trả JSON duy nhất:\n"
        '{"targets":[{"span":"...","pos":"noun|verb"}],"link_prefix_sentence2":""}\n'
        "Không giải thích."
    )
    raw = _invoke_upgrade_llm(prompt, model=ollama_model, temperature=0.1)
    data = _json_upgrade_as_dict(_extract_json_upgrade(raw))
    targets = data.get("targets", [])
    if not isinstance(targets, list):
        targets = []
    link_prefix = data.get("link_prefix_sentence2", "") or ""
    if not isinstance(link_prefix, str):
        link_prefix = ""
    link_prefix = link_prefix.strip()
    out = []
    for t in targets:
        if not isinstance(t, dict):
            continue
        sp = str(t.get("span", "")).strip()
        pos = str(t.get("pos", "")).strip().lower()
        if not sp or (sp not in sentence_a and sp not in sentence_b):
            continue
        if pos not in ("noun", "verb", "danh từ", "động từ"):
            pos = "word"
        out.append({"span": sp, "pos": pos})
    return out, link_prefix


def ask_llm_synonyms_for_nv_spans(sentence: str, targets: list, ollama_model: str):
    if not targets:
        return {}
    lines = "\n".join([f'- [{t.get("pos","?")}] {t["span"]}' for t in targets])
    prompt = (
        f'Cho câu: "{sentence}"\n'
        "Với từng mục sau (đã là danh từ/động từ trong câu), đề xuất tối đa 4 **đồng nghĩa hoặc cụm thay thế** phù hợp ngữ cảnh, trang trọng hơn, "
        "không đổi nghĩa chính. Không làm dài câu quá đáng.\n"
        f"{lines}\n"
        "Trả JSON duy nhất: {\"synonyms\":{\"<span đúng như trên>\":[\"gợi ý 1\", ...]}}\n"
        "Không giải thích."
    )
    raw = _invoke_upgrade_llm(prompt, model=ollama_model, temperature=0.2)
    data = _json_upgrade_as_dict(_extract_json_upgrade(raw))
    syn = data.get("synonyms", {})
    return syn if isinstance(syn, dict) else {}


def ask_llm_synonyms_for_nv_spans_pair(sentence_a: str, sentence_b: str, targets: list, ollama_model: str):
    if not targets:
        return {}
    lines = "\n".join([f'- [{t.get("pos","?")}] {t["span"]}' for t in targets])
    prompt = (
        "Cho hai câu liên tiếp:\n"
        f'Câu 1: "{sentence_a}"\n'
        f'Câu 2: "{sentence_b}"\n'
        "Với từng mục span (xuất hiện trong một trong hai câu), đề xuất tối đa 4 **đồng nghĩa hoặc cụm thay thế** phù hợp ngữ cảnh toàn khối hai câu, trang trọng hơn, "
        "không đổi nghĩa chính.\n"
        f"{lines}\n"
        "Trả JSON duy nhất: {\"synonyms\":{\"<span đúng như trên>\":[\"gợi ý 1\", ...]}}\n"
        "Không giải thích."
    )
    raw = _invoke_upgrade_llm(prompt, model=ollama_model, temperature=0.2)
    data = _json_upgrade_as_dict(_extract_json_upgrade(raw))
    syn = data.get("synonyms", {})
    return syn if isinstance(syn, dict) else {}


def _apply_link_prefix_sentence2(sentence_b: str, prefix: str) -> str:
    p = (prefix or "").strip()
    if not p:
        return sentence_b
    return p.rstrip() + " " + sentence_b.lstrip()


def _lookup_synonyms(syn_map: dict, span: str):
    if span in syn_map:
        v = syn_map[span]
    else:
        sn = _normalize_token(span.replace(" ", "_"))
        v = None
        for k, val in syn_map.items():
            if _normalize_token(str(k).replace(" ", "_")) == sn:
                v = val
                break
    if v is None:
        return []
    if isinstance(v, str):
        return [v]
    if isinstance(v, list):
        return [str(x).strip() for x in v if str(x).strip()]
    return []


def upgrade_one_sentence_controlled(
    sentence: str,
    vocab_next: set[str],
    max_replacements: int,
    min_semantic_f1: float,
    max_length_ratio: float,
    ollama_model: str,
):
    sent_orig = sentence.strip()
    temp = sent_orig
    log = {"sentence_original": sent_orig, "steps": []}

    if not sent_orig:
        return {"text": "", "applied": [], "touched_words": 0, "log": log}

    n_words_s = len(preprocess_vi_eval(sent_orig).split())
    max_targets = max(4, min(12, n_words_s // 2 + 1))

    targets = ask_llm_nv_targets(sent_orig, ollama_model, max_targets=max_targets)
    syn_map = ask_llm_synonyms_for_nv_spans(sent_orig, targets, ollama_model)

    uniq_spans = []
    seen_sp = set()
    for sp in sorted([t["span"] for t in targets], key=len, reverse=True):
        if sp and sp not in seen_sp:
            seen_sp.add(sp)
            uniq_spans.append(sp)
    spans_ordered = uniq_spans
    applied = []
    touched = 0

    for span in spans_ordered:
        if len(applied) >= max_replacements:
            break
        if span not in temp:
            continue
        candidates = _lookup_synonyms(syn_map, span)
        filtered = [c for c in candidates if _replacement_allowed_for_next_grade(c, vocab_next)]
        if not filtered:
            continue

        best_candidate = None
        best_pair = (-1.0, 99.0)
        for cand in filtered:
            trial = replace_span_once(temp, span, cand)
            if trial == temp:
                continue
            f1, lr = similarity_candidate_vs_reference(trial, sent_orig)
            if f1 >= min_semantic_f1 and lr <= max_length_ratio:
                if f1 > best_pair[0]:
                    best_candidate = cand
                    best_pair = (f1, lr)

        if best_candidate is None:
            log["steps"].append({"span": span, "reason": "no_syn_passes_threshold_or_vocab"})
            continue

        trial_final = replace_span_once(temp, span, best_candidate)
        tw = len(preprocess_vi_eval(span).split())
        temp = trial_final
        touched += tw
        applied.append(
            {
                "span": span,
                "to": best_candidate,
                "bertscore_f1": best_pair[0],
                "length_ratio": best_pair[1],
            }
        )
        log["steps"].append({"span": span, "to": best_candidate, "accepted": True})

    return {"text": temp, "applied": applied, "touched_words": touched, "log": log}


def upgrade_sentence_pair_controlled(
    sentence_a: str,
    sentence_b: str,
    vocab_next: set[str],
    max_replacements: int,
    min_semantic_f1: float,
    max_length_ratio: float,
    ollama_model: str,
):
    sa_orig = sentence_a.strip()
    sb_orig = sentence_b.strip()
    temp_a, temp_b = sa_orig, sb_orig
    ref_pair = sa_orig + " " + sb_orig
    log = {
        "pair": True,
        "sentence_a_original": sa_orig,
        "sentence_b_original": sb_orig,
        "link_prefix_proposed": "",
        "link_prefix_applied": "",
        "steps": [],
    }

    if not sa_orig or not sb_orig:
        return {
            "text_a": temp_a,
            "text_b": temp_b,
            "applied": [],
            "touched_words": 0,
            "log": log,
        }

    wa = len(preprocess_vi_eval(sa_orig).split())
    wb = len(preprocess_vi_eval(sb_orig).split())
    max_targets = max(4, min(14, (wa + wb) // 2 + 3))

    targets, link_prefix = ask_llm_nv_targets_pair(sa_orig, sb_orig, ollama_model, max_targets=max_targets)
    log["link_prefix_proposed"] = link_prefix
    syn_map = ask_llm_synonyms_for_nv_spans_pair(sa_orig, sb_orig, targets, ollama_model)

    seen_sp = set()
    uniq_spans = []
    for sp in sorted([t["span"] for t in targets], key=len, reverse=True):
        if sp and sp not in seen_sp:
            seen_sp.add(sp)
            uniq_spans.append(sp)

    applied = []
    touched = 0
    nv_count = 0

    for span in uniq_spans:
        if nv_count >= max_replacements:
            break
        candidates = _lookup_synonyms(syn_map, span)
        filtered = [c for c in candidates if _replacement_allowed_for_next_grade(c, vocab_next)]
        if not filtered:
            continue

        trial_pair = None
        best_buf = None
        best_candidate = None
        best_pair = (-1.0, 99.0)

        if span in temp_a:
            for cand in filtered:
                ta = replace_span_once(temp_a, span, cand)
                if ta == temp_a:
                    continue
                trial_pair = ta + " " + temp_b
                f1, lr = similarity_candidate_vs_reference(trial_pair, ref_pair)
                if f1 >= min_semantic_f1 and lr <= max_length_ratio and f1 > best_pair[0]:
                    best_buf = "a"
                    best_candidate = cand
                    best_pair = (f1, lr)

        if span in temp_b:
            for cand in filtered:
                tb = replace_span_once(temp_b, span, cand)
                if tb == temp_b:
                    continue
                trial_pair = temp_a + " " + tb
                f1, lr = similarity_candidate_vs_reference(trial_pair, ref_pair)
                if f1 >= min_semantic_f1 and lr <= max_length_ratio and f1 > best_pair[0]:
                    best_buf = "b"
                    best_candidate = cand
                    best_pair = (f1, lr)

        if best_buf is None or best_candidate is None:
            log["steps"].append({"span": span, "reason": "no_syn_passes_threshold_or_vocab"})
            continue

        if best_buf == "a":
            temp_a = replace_span_once(temp_a, span, best_candidate)
        else:
            temp_b = replace_span_once(temp_b, span, best_candidate)
        nv_count += 1
        tw = len(preprocess_vi_eval(span).split())
        touched += tw
        applied.append(
            {
                "span": span,
                "to": best_candidate,
                "sentence": best_buf,
                "bertscore_f1": best_pair[0],
                "length_ratio": best_pair[1],
            }
        )
        log["steps"].append({"span": span, "sentence": best_buf, "to": best_candidate, "accepted": True})

    if link_prefix:
        trial_b = _apply_link_prefix_sentence2(temp_b, link_prefix)
        f1, lr = similarity_candidate_vs_reference(temp_a + " " + trial_b, ref_pair)
        vocab_ok = _replacement_allowed_for_next_grade(link_prefix.rstrip(",").strip(), vocab_next) or _replacement_allowed_for_next_grade(
            link_prefix, vocab_next
        )
        if vocab_ok and f1 >= min_semantic_f1 and lr <= max_length_ratio:
            tw_link = len(preprocess_vi_eval(link_prefix).split())
            temp_b = trial_b
            touched += tw_link
            log["link_prefix_applied"] = link_prefix
            applied.append(
                {
                    "kind": "link_prefix_sentence2",
                    "to": link_prefix,
                    "bertscore_f1": f1,
                    "length_ratio": lr,
                }
            )
            log["steps"].append({"link_prefix_sentence2": link_prefix, "accepted": True})
        else:
            log["steps"].append({"link_prefix_sentence2": link_prefix, "accepted": False, "reason": "threshold_or_vocab"})

    return {"text_a": temp_a, "text_b": temp_b, "applied": applied, "touched_words": touched, "log": log}


def upgrade_summary_difficulty_medium(
    summary: str,
    target_grade: int,
    max_upgrade_ratio: float = DIFF_UPGRADE_MAX_RATIO,
    min_semantic_f1: float = DIFF_MIN_BERT_F1,
    max_length_ratio: float = DIFF_MAX_LENGTH_RATIO,
    ollama_model: str = "qwen2.5:7b",
):
    """Theo cặp hai câu liên tiếp: LLM xét ngữ cảnh + gợi ý trạng ngữ đầu câu sau; DN/ĐT thay trong từng câu theo ngưỡng BERT/độ dài."""
    if not isinstance(summary, str) or not summary.strip():
        return {
            "upgraded_summary": summary,
            "applied_upgrades": [],
            "sentence_logs": [],
            "metrics": {},
            "skipped_reason": "empty_summary",
        }

    if target_grade >= 5:
        return {
            "upgraded_summary": summary.strip(),
            "applied_upgrades": [],
            "sentence_logs": [],
            "metrics": {"note": "grade_5_kept_unchanged"},
            "skipped_reason": None,
        }

    next_g = target_grade + 1
    vocab_next = load_grade_vocab(next_g)
    if not vocab_next:
        return {
            "upgraded_summary": summary.strip(),
            "applied_upgrades": [],
            "sentence_logs": [],
            "metrics": {},
            "skipped_reason": f"no_vocab_grade_{next_g}",
        }

    sentences = split_sentences_for_upgrade(summary.strip())
    n_words_global = len(preprocess_vi_eval(summary).split())
    max_touch_words = n_words_global * max_upgrade_ratio
    touched_global = 0
    out_parts = []
    all_applied = []
    sentence_logs = []

    i = 0
    while i < len(sentences):
        wa_i = len(preprocess_vi_eval(sentences[i]).split())
        wb_next = len(preprocess_vi_eval(sentences[i + 1]).split()) if i + 1 < len(sentences) else 0
        budget_words = max(0.0, max_touch_words - touched_global)

        if budget_words < 1:
            if i + 1 < len(sentences):
                out_parts.extend([sentences[i], sentences[i + 1]])
                sentence_logs.append({"pair": True, "skipped": "global_word_budget"})
                i += 2
            else:
                out_parts.append(sentences[i])
                sentence_logs.append({"sentence_in": sentences[i], "skipped": "global_word_budget"})
                i += 1
            continue

        if i + 1 < len(sentences):
            max_rep = max(1, int((wa_i + wb_next) * max_upgrade_ratio) + 1)
            one = upgrade_sentence_pair_controlled(
                sentence_a=sentences[i],
                sentence_b=sentences[i + 1],
                vocab_next=vocab_next,
                max_replacements=max_rep,
                min_semantic_f1=min_semantic_f1,
                max_length_ratio=max_length_ratio,
                ollama_model=ollama_model,
            )
            tw = one["touched_words"]
            if touched_global + tw > max_touch_words + 1e-6:
                out_parts.extend([sentences[i], sentences[i + 1]])
                sentence_logs.append({"pair": True, "skipped": "would_exceed_global_ratio"})
                i += 2
                continue

            out_parts.append(one["text_a"])
            out_parts.append(one["text_b"])
            touched_global += tw
            all_applied.extend(one["applied"])
            sentence_logs.append(one["log"])
            i += 2
        else:
            max_rep = max(1, int(wa_i * max_upgrade_ratio) + 1)
            one = upgrade_one_sentence_controlled(
                sentence=sentences[i],
                vocab_next=vocab_next,
                max_replacements=max_rep,
                min_semantic_f1=min_semantic_f1,
                max_length_ratio=max_length_ratio,
                ollama_model=ollama_model,
            )
            tw = one["touched_words"]
            if touched_global + tw > max_touch_words + 1e-6:
                out_parts.append(sentences[i])
                sentence_logs.append({"sentence_in": sentences[i], "skipped": "would_exceed_global_ratio"})
                i += 1
                continue

            out_parts.append(one["text"])
            touched_global += tw
            all_applied.extend(one["applied"])
            sentence_logs.append(one["log"])
            i += 1

    upgraded = " ".join(out_parts).strip()
    f1_all, lr_all = similarity_candidate_vs_reference(upgraded, summary.strip())

    if f1_all < min_semantic_f1 or lr_all > max_length_ratio:
        return {
            "upgraded_summary": summary.strip(),
            "applied_upgrades": [],
            "sentence_logs": sentence_logs,
            "metrics": {
                "bertscore_f1": f1_all,
                "length_ratio": lr_all,
                "candidate_preview": upgraded,
            },
            "skipped_reason": "whole_text_constraints_failed_reverted",
        }

    return {
        "upgraded_summary": upgraded,
        "applied_upgrades": all_applied,
        "sentence_logs": sentence_logs,
        "metrics": {
            "bertscore_f1": f1_all,
            "length_ratio": lr_all,
            "upgrade_ratio_words_est": touched_global / max(n_words_global, 1),
            "max_upgrade_ratio_cap": max_upgrade_ratio,
            "touched_word_estimate": touched_global,
            "total_words": n_words_global,
            "next_grade_vocab_used": next_g,
        },
        "skipped_reason": None,
    }


# Gọi thử (cần cell metrics: preprocess_vi_eval, compute_bertscore_eval, load_grade_vocab, _normalize_token)
demo_txt = "Trong quá trình học tập, mỗi học sinh cần có ý thức và trách nhiệm đối với việc phát triển bản thân. Việc tiếp thu kiến thức không chỉ giúp nâng cao năng lực mà còn góp phần hình thành tư duy logic và khả năng phân tích vấn đề. Bên cạnh đó, môi trường giáo dục đóng vai trò quan trọng trong việc định hướng và hỗ trợ học sinh đạt được mục tiêu của mình. Nếu biết tận dụng cơ hội và không ngừng nỗ lực, các em sẽ từng bước hoàn thiện kỹ năng và xây dựng nền tảng vững chắc cho tương lai."
print(upgrade_summary_difficulty_medium(demo_txt, target_grade=4, max_upgrade_ratio=0.3, min_semantic_f1=0.9, max_length_ratio=1.2, ollama_model="qwen2.5:7b"))


C:\Users\minhv\AppData\Local\Temp\ipykernel_2792\2894567471.py:113: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  _UPGRADE_LLM[key] = ChatOllama(


{'upgraded_summary': 'Trong quá trình học tập, mỗi học sinh cần có ý thức và trách nhiệm đối với việc phát triển bản thân. Việc tiếp thu kiến thức không chỉ giúp nâng cao năng lực mà còn góp phần hình thành tư duy logic và khả năng phân tích vấn đề. Bên cạnh đó, cơ sở giáo dục đóng vai trò quan trọng trong việc hướng dẫn và trợ giúp học sinh lấy lại mục tiêu của mình. Nếu biết tận dụng cơ hội và không ngừng nỗ lực, các em sẽ từng bước hoàn thiện năng lực và xây dựng bệ phóng vững chắc cho tương lai.', 'applied_upgrades': [{'span': 'môi trường giáo dục', 'to': 'cơ sở giáo dục', 'sentence': 'a', 'bertscore_f1': 0.9926377534866333, 'length_ratio': 1.0}, {'span': 'nền tảng vững chắc', 'to': 'bệ phóng vững chắc', 'sentence': 'b', 'bertscore_f1': 0.9810574650764465, 'length_ratio': 1.0227272727272727}, {'span': 'định hướng', 'to': 'hướng dẫn', 'sentence': 'a', 'bertscore_f1': 0.9754205942153931, 'length_ratio': 1.0227272727272727}, {'span': 'đạt được', 'to': 'lấy lại', 'sentence': 'a', 'bert

Trong quá trình học tập, mỗi học sinh cần có ý thức và trách nhiệm đối với việc phát triển bản thân. Việc tiếp thu kiến thức không chỉ giúp nâng cao năng lực mà còn góp phần hình thành tư duy logic và khả năng phân tích vấn đề. Bên cạnh đó, môi trường giáo dục đóng vai trò quan trọng trong việc định hướng và hỗ trợ học sinh đạt được mục tiêu của mình. Nếu biết tận dụng cơ hội và không ngừng nỗ lực, các em sẽ từng bước hoàn thiện kỹ năng và xây dựng nền tảng vững chắc cho tương lai.


Mỗi học sinh cần có ý thức và trách nhiệm đối với việc phát triển bản thân. Việc tiếp thu kiến thức không chỉ giúp nâng cao năng lực mà còn góp phần hình thành tư duy logic và khả năng phân tích vấn đề. Môi trường giáo dục đóng vai trò quan trọng. Các em sẽ từng bước hoàn thiện kỹ năng và xây dựng nền tảng vững chắc cho tương lai.

Trong quá trình học tập, mỗi sinh viên cần có ý thức và trách nhiệm đối với việc tăng cường bản thân. Việc tiếp thu nhận thức không chỉ giúp thúc đẩy năng lực mà còn góp phần hình thành lý luận và khả năng phân tích vấn đề. Bên cạnh đó, thế giới học đường đóng vai trò quan trọng trong việc hướng dẫn và trợ giúp học sinh thực hiện mục tiêu mục tiêu của mình. Nếu biết tận dụng cơ hội và không ngừng nỗ lực, các em sẽ tiến bộ từng ngày kỹ năng và tạo dựng nền tảng vững chắc cho tương lai.